# CPO (Certified Pre-Owned) Vehicle Market Analysis

Analyses inventory, pricing, age/mileage profiles, and value metrics across Singapore's official CPO programmes.

**Sources scraped:** Das WeltAuto (VW/Skoda), Eurokars (Volvo/Porsche), Toyota, Dickson, CarChoice, Cycle & Carriage, Sim Mee Motors (Audi), IC Pre-owned (BMW/MINI)

**Key questions:**
- Which brands / dealers have the most inventory?
- How does CPO pricing compare across sources?
- Which listings offer the best value (lowest price-per-year-of-age, price-per-km)?
- How is the total CPO market trending over time?

In [ ]:
import glob
import json
import re
from datetime import datetime, date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR if (NB_DIR / "data").exists() else NB_DIR.parent
CPO_DIR = PROJECT_ROOT / "data" / "cpo"
LATEST_FILE = CPO_DIR / "latest.json"

TODAY = date.today()
print(f"Project root: {PROJECT_ROOT}")
print(f"CPO data dir: {CPO_DIR}")
print(f"Today: {TODAY}")

## 1. Load & Parse Data

In [ ]:
# --- Parsing helpers ---

_YEAR_RE = re.compile(r"^(\d{4})\s+(.+)")
_KNOWN_BRANDS = {
    "Volkswagen", "VW", "Volvo", "Toyota", "Audi", "BMW", "MINI",
    "Mercedes-Benz", "Mercedes", "Porsche", "Skoda", "Honda", "Hyundai",
    "Kia", "Lexus", "Mazda", "Nissan", "Subaru", "Suzuki", "Mitsubishi",
    "Jaguar", "Land", "Ford", "Renault", "Peugeot", "Citroen", "Seat",
    "Cupra", "Genesis", "BYD", "Tesla", "Alfa",
}

def extract_brand(listing: dict) -> str:
    """Return normalised brand: use explicit field if available, else parse title."""
    if listing.get("brand"):
        return listing["brand"].strip().title()
    title = (listing.get("title") or "").strip()
    # Strip leading year
    m = _YEAR_RE.match(title)
    if m:
        title = m.group(2)
    return title.split()[0] if title else "Unknown"


_MILEAGE_RE = re.compile(r"([\d,]+)")

def parse_mileage(text) -> float | None:
    if not text:
        return None
    m = _MILEAGE_RE.search(str(text))
    return float(m.group(1).replace(",", "")) if m else None


_DATE_FORMATS = [
    "%d/%m/%Y",
    "%d/%m/%y",
    "%d-%m-%Y",
    "%Y-%m-%d",
    "%d %b %Y",
    "%d-%b-%Y",
    "%d-%b-%y",
    "%d/%b/%Y",
    "%d %B %Y",
]

def parse_reg_date(text) -> date | None:
    if not text:
        return None
    text = str(text).strip()
    # Handle ambiguous d/m/Y vs m/d/Y: Singapore convention is always DD/MM
    for fmt in _DATE_FORMATS:
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            continue
    return None


print("Helpers ready.")

In [ ]:
with open(LATEST_FILE) as f:
    raw = json.load(f)

print(f"Snapshot date : {raw['date']}")
print(f"Total listings: {raw['total_listings']}")
print("\nPer-source counts:")
for src, info in raw["site_results"].items():
    print(f"  {src:<20} {info['count']:>4}  ({info['status']})")

In [ ]:
rows = []
for lst in raw["listings"]:
    if lst.get("listing_type") != "listing":
        continue
    reg_date = parse_reg_date(lst.get("reg_date"))
    age_years = (TODAY - reg_date).days / 365.25 if reg_date else None
    mileage_km = parse_mileage(lst.get("mileage"))
    rows.append(
        {
            "source": lst["source"],
            "title": lst.get("title", ""),
            "url": lst.get("url", ""),
            "brand": extract_brand(lst),
            "model": lst.get("model") or "",
            "price": lst.get("price"),
            "mileage_km": mileage_km,
            "reg_date": reg_date,
            "age_years": age_years,
            "scraped_date": lst.get("scraped_date"),
        }
    )

df = pd.DataFrame(rows)

# Ensure numeric dtypes (source JSON may mix int/None/str)
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["age_years"] = pd.to_numeric(df["age_years"], errors="coerce")
df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")

# Derived metrics
df["price_per_age_year"] = np.where(
    df["age_years"].notna() & (df["age_years"] > 0) & df["price"].notna(),
    df["price"] / df["age_years"],
    np.nan,
)
df["price_per_10k_km"] = np.where(
    df["mileage_km"].notna() & (df["mileage_km"] > 0) & df["price"].notna(),
    df["price"] / (df["mileage_km"] / 10_000),
    np.nan,
)
df["annual_mileage_km"] = np.where(
    df["mileage_km"].notna() & df["age_years"].notna() & (df["age_years"] > 0),
    df["mileage_km"] / df["age_years"],
    np.nan,
)

# Basic sanity filter: price > 0, age >= 0
df["suspicious"] = (
    (df["age_years"].notna() & (df["age_years"] < 0))
    | (df["price"].notna() & (df["price"] <= 0))
)
clean = df[~df["suspicious"]].copy()

print(f"Total rows  : {len(df)}")
print(f"Clean rows  : {len(clean)}")
print(f"Suspicious  : {df['suspicious'].sum()}")
print(f"\nField coverage (clean):")
for col in ["price", "mileage_km", "reg_date", "age_years"]:
    pct = clean[col].notna().mean() * 100
    print(f"  {col:<20} {pct:.0f}%")


## 2. Inventory Overview

In [ ]:
SOURCE_LABELS = {
    "das_weltauto": "Das WeltAuto (VW/Skoda)",
    "eurokars": "Eurokars (Volvo/Porsche)",
    "toyota": "Toyota",
    "dickson": "Dickson",
    "carchoice": "CarChoice",
    "cycle_carriage": "Cycle & Carriage",
    "skoda": "Skoda",
    "sim_mee_motors": "Sim Mee Motors (Audi)",
    "ic_preowned": "IC Pre-owned (BMW/MINI)",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Listings by source
src_counts = clean["source"].value_counts()
src_labels = [SOURCE_LABELS.get(s, s) for s in src_counts.index]
axes[0].barh(src_labels[::-1], src_counts.values[::-1], color=sns.color_palette("tab10", len(src_counts))[::-1])
axes[0].set_xlabel("Number of listings")
axes[0].set_title("CPO Listings by Dealer")
for i, v in enumerate(src_counts.values[::-1]):
    axes[0].text(v + 0.5, i, str(v), va="center", fontsize=9)

# Listings by brand (top 12)
brand_counts = clean["brand"].value_counts().head(12)
axes[1].barh(brand_counts.index[::-1], brand_counts.values[::-1], color=sns.color_palette("tab20", len(brand_counts))[::-1])
axes[1].set_xlabel("Number of listings")
axes[1].set_title("CPO Listings by Brand (top 12)")
for i, v in enumerate(brand_counts.values[::-1]):
    axes[1].text(v + 0.5, i, str(v), va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Price Distribution

In [ ]:
priced = clean[clean["price"].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall price histogram
axes[0].hist(priced["price"] / 1000, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(priced["price"].median() / 1000, color="red", linestyle="--", label=f'Median: ${priced["price"].median()/1000:.0f}K')
axes[0].axvline(priced["price"].mean() / 1000, color="orange", linestyle="--", label=f'Mean: ${priced["price"].mean()/1000:.0f}K')
axes[0].set_xlabel("Price (SGD '000)")
axes[0].set_ylabel("Count")
axes[0].set_title("CPO Price Distribution")
axes[0].legend()

# Price by source (box plot)
src_order = clean.groupby("source")["price"].median().sort_values().index
src_label_order = [SOURCE_LABELS.get(s, s) for s in src_order]
plot_df = priced.copy()
plot_df["source_label"] = plot_df["source"].map(lambda s: SOURCE_LABELS.get(s, s))
sns.boxplot(
    data=plot_df,
    y="source_label",
    x="price",
    order=src_label_order,
    ax=axes[1],
    palette="tab10",
    showfliers=False,
)
axes[1].set_xlabel("Price (SGD)")
axes[1].set_ylabel("")
axes[1].set_title("Price Range by Dealer")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))

plt.tight_layout()
plt.show()

print(f"\nPrice summary (SGD):")
print(priced["price"].describe().apply(lambda x: f"${x:,.0f}"))

In [ ]:
# Price by brand (top brands with ≥ 5 listings)
brand_enough = clean["brand"].value_counts()
brand_enough = brand_enough[brand_enough >= 5].index
brand_plot_df = priced[priced["brand"].isin(brand_enough)].copy()
brand_order = brand_plot_df.groupby("brand")["price"].median().sort_values().index

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=brand_plot_df,
    y="brand",
    x="price",
    order=brand_order,
    ax=ax,
    palette="tab20",
    showfliers=False,
)
ax.set_xlabel("Price (SGD)")
ax.set_ylabel("")
ax.set_title("CPO Price by Brand (brands with ≥5 listings, excl. outliers)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
plt.tight_layout()
plt.show()

## 4. Age & Mileage Profiles

In [ ]:
aged = clean[clean["age_years"].notna()].copy()
miled = clean[clean["mileage_km"].notna() & (clean["mileage_km"] < 400_000)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(aged["age_years"], bins=25, color="teal", edgecolor="white")
axes[0].axvline(aged["age_years"].median(), color="red", linestyle="--",
                label=f'Median: {aged["age_years"].median():.1f} yr')
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Count")
axes[0].set_title("CPO Car Age Distribution")
axes[0].legend()

axes[1].hist(miled["mileage_km"] / 1000, bins=30, color="coral", edgecolor="white")
axes[1].axvline(miled["mileage_km"].median() / 1000, color="red", linestyle="--",
                label=f'Median: {miled["mileage_km"].median()/1000:.0f}K km')
axes[1].set_xlabel("Mileage ('000 km)")
axes[1].set_ylabel("Count")
axes[1].set_title("CPO Mileage Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Age summary (years):")
print(aged["age_years"].describe().apply(lambda x: f"{x:.2f}"))
print("\nMileage summary (km):")
print(miled["mileage_km"].describe().apply(lambda x: f"{x:,.0f}"))

In [ ]:
# Age vs Mileage scatter, coloured by source
scatter_df = clean[
    clean["age_years"].notna()
    & clean["mileage_km"].notna()
    & (clean["mileage_km"] < 400_000)
    & (clean["age_years"] >= 0)
].copy()

fig, ax = plt.subplots(figsize=(11, 6))
sources_in_scatter = scatter_df["source"].unique()
palette = sns.color_palette("tab10", len(sources_in_scatter))
for color, src in zip(palette, sources_in_scatter):
    sub = scatter_df[scatter_df["source"] == src]
    ax.scatter(sub["age_years"], sub["mileage_km"] / 1000,
               label=SOURCE_LABELS.get(src, src), alpha=0.7, s=40, color=color)

# Expected mileage line: 15,000 km/yr (typical SG)
age_range = np.linspace(0, scatter_df["age_years"].max(), 100)
ax.plot(age_range, age_range * 15, "k--", linewidth=1, label="15,000 km/yr (typical)")

ax.set_xlabel("Age (years)")
ax.set_ylabel("Mileage ('000 km)")
ax.set_title("Age vs Mileage — CPO Listings")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 5. Price vs Age & Price vs Mileage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price vs Age
pa_df = clean[clean["age_years"].notna() & clean["price"].notna() & (clean["age_years"] >= 0)].copy()
palette_src = {s: c for s, c in zip(pa_df["source"].unique(), sns.color_palette("tab10", pa_df["source"].nunique()))}
for src, color in palette_src.items():
    sub = pa_df[pa_df["source"] == src]
    axes[0].scatter(sub["age_years"], sub["price"] / 1000,
                    label=SOURCE_LABELS.get(src, src), alpha=0.65, s=35, color=color)
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Price (SGD '000)")
axes[0].set_title("Price vs Age")
axes[0].legend(fontsize=7)

# Price vs Mileage
pm_df = clean[
    clean["mileage_km"].notna()
    & clean["price"].notna()
    & (clean["mileage_km"] < 400_000)
].copy()
for src, color in palette_src.items():
    sub = pm_df[pm_df["source"] == src]
    axes[1].scatter(sub["mileage_km"] / 1000, sub["price"] / 1000,
                    label=SOURCE_LABELS.get(src, src), alpha=0.65, s=35, color=color)
axes[1].set_xlabel("Mileage ('000 km)")
axes[1].set_ylabel("Price (SGD '000)")
axes[1].set_title("Price vs Mileage")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## 6. Value Metrics

In [ ]:
# Price per year of age: lower → more car per dollar of age premium absorbed
ppa = clean[
    clean["price_per_age_year"].notna()
    & (clean["age_years"] >= 1)  # exclude near-new
].copy()

# Price per 10k km: lower → more car per dollar of wear absorbed
ppk = clean[
    clean["price_per_10k_km"].notna()
    & (clean["mileage_km"] >= 5_000)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ppa["price_per_age_year"] / 1000, bins=30, color="mediumseagreen", edgecolor="white")
axes[0].axvline(ppa["price_per_age_year"].median() / 1000, color="red", linestyle="--",
                label=f'Median: ${ppa["price_per_age_year"].median()/1000:.0f}K/yr')
axes[0].set_xlabel("Price per year of age (SGD '000)")
axes[0].set_title("Price per Year of Age Distribution")
axes[0].legend()

axes[1].hist(ppk["price_per_10k_km"] / 1000, bins=30, color="mediumpurple", edgecolor="white")
axes[1].axvline(ppk["price_per_10k_km"].median() / 1000, color="red", linestyle="--",
                label=f'Median: ${ppk["price_per_10k_km"].median()/1000:.0f}K/10k km')
axes[1].set_xlabel("Price per 10,000 km (SGD '000)")
axes[1].set_title("Price per 10k km Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Source / Dealer Comparison

In [ ]:
dealer_stats = (
    clean.groupby("source")
    .agg(
        count=("title", "count"),
        median_price=("price", "median"),
        median_age=("age_years", "median"),
        median_mileage=("mileage_km", "median"),
        median_ppa=("price_per_age_year", "median"),
        median_ppk=("price_per_10k_km", "median"),
    )
    .reset_index()
)
dealer_stats["dealer"] = dealer_stats["source"].map(lambda s: SOURCE_LABELS.get(s, s))
dealer_stats = dealer_stats.sort_values("median_price")

print("Dealer summary table:")
display_cols = ["dealer", "count", "median_price", "median_age", "median_mileage", "median_ppa"]
ds_display = dealer_stats[display_cols].copy()
ds_display["median_price"] = ds_display["median_price"].apply(lambda x: f"${x:,.0f}" if pd.notna(x) else "—")
ds_display["median_age"] = ds_display["median_age"].apply(lambda x: f"{x:.1f} yr" if pd.notna(x) else "—")
ds_display["median_mileage"] = ds_display["median_mileage"].apply(lambda x: f"{x/1000:.0f}K km" if pd.notna(x) else "—")
ds_display["median_ppa"] = ds_display["median_ppa"].apply(lambda x: f"${x/1000:.0f}K/yr" if pd.notna(x) else "—")
ds_display.columns = ["Dealer", "Listings", "Median Price", "Median Age", "Median Mileage", "Median $/yr of age"]
print(ds_display.to_string(index=False))

## 8. Top Value Picks

In [ ]:
print("## Top 20 — Lowest Price per Year of Age (cars ≥1 yr old)\n")
print("Lower = you're paying less per year of age the car has absorbed.\n")
top_ppa = ppa.nsmallest(20, "price_per_age_year")[
    ["title", "source", "price", "age_years", "mileage_km", "price_per_age_year"]
].copy()
top_ppa["price"] = top_ppa["price"].apply(lambda x: f"${x:,.0f}")
top_ppa["age_years"] = top_ppa["age_years"].apply(lambda x: f"{x:.1f} yr")
top_ppa["mileage_km"] = top_ppa["mileage_km"].apply(lambda x: f"{x/1000:.0f}K km" if pd.notna(x) else "—")
top_ppa["price_per_age_year"] = top_ppa["price_per_age_year"].apply(lambda x: f"${x/1000:.0f}K/yr")
top_ppa["source"] = top_ppa["source"].map(lambda s: SOURCE_LABELS.get(s, s))
top_ppa.columns = ["Title", "Dealer", "Price", "Age", "Mileage", "$/yr of age"]
print(top_ppa.to_string(index=False))

In [ ]:
print("## Top 20 — Lowest Price per 10k km (cars with ≥5k km)\n")
print("Lower = the seller has absorbed more wear per dollar of asking price.\n")
top_ppk = ppk.nsmallest(20, "price_per_10k_km")[
    ["title", "source", "price", "mileage_km", "age_years", "price_per_10k_km"]
].copy()
top_ppk["price"] = top_ppk["price"].apply(lambda x: f"${x:,.0f}")
top_ppk["mileage_km"] = top_ppk["mileage_km"].apply(lambda x: f"{x/1000:.0f}K km")
top_ppk["age_years"] = top_ppk["age_years"].apply(lambda x: f"{x:.1f} yr" if pd.notna(x) else "—")
top_ppk["price_per_10k_km"] = top_ppk["price_per_10k_km"].apply(lambda x: f"${x/1000:.0f}K/10k km")
top_ppk["source"] = top_ppk["source"].map(lambda s: SOURCE_LABELS.get(s, s))
top_ppk.columns = ["Title", "Dealer", "Price", "Mileage", "Age", "$/10k km"]
print(top_ppk.to_string(index=False))

In [ ]:
# Composite value score (lower = better value)
# Rank on: price_per_age_year + price_per_10k_km (both normalised 0-1)
combo = clean[
    clean["price_per_age_year"].notna()
    & clean["price_per_10k_km"].notna()
    & (clean["age_years"] >= 1)
    & (clean["mileage_km"] >= 5_000)
].copy()

for col in ["price_per_age_year", "price_per_10k_km"]:
    mn, mx = combo[col].min(), combo[col].max()
    combo[f"{col}_norm"] = (combo[col] - mn) / (mx - mn)

combo["value_score"] = (combo["price_per_age_year_norm"] + combo["price_per_10k_km_norm"]) / 2

print("## Top 20 — Best Composite Value Score (age + mileage)\n")
print("Score 0 = best value, 1 = worst.\n")
top_combo = combo.nsmallest(20, "value_score")[
    ["title", "source", "price", "age_years", "mileage_km", "value_score"]
].copy()
top_combo["price"] = top_combo["price"].apply(lambda x: f"${x:,.0f}")
top_combo["age_years"] = top_combo["age_years"].apply(lambda x: f"{x:.1f} yr")
top_combo["mileage_km"] = top_combo["mileage_km"].apply(lambda x: f"{x/1000:.0f}K km")
top_combo["value_score"] = top_combo["value_score"].apply(lambda x: f"{x:.2f}")
top_combo["source"] = top_combo["source"].map(lambda s: SOURCE_LABELS.get(s, s))
top_combo.columns = ["Title", "Dealer", "Price", "Age", "Mileage", "Score"]
print(top_combo.to_string(index=False))

## 9. Historical Trends

In [ ]:
# Load all dated snapshots to build a time series
snapshot_files = sorted(CPO_DIR.glob("20*.json"))
print(f"Found {len(snapshot_files)} dated snapshots")
print(f"Date range: {snapshot_files[0].stem} → {snapshot_files[-1].stem}")

ts_rows = []
for fp in snapshot_files:
    with open(fp) as f:
        snap = json.load(f)
    snap_date = snap.get("date", fp.stem)
    listings_raw = [l for l in snap.get("listings", []) if l.get("listing_type") == "listing"]
    prices = [l["price"] for l in listings_raw if l.get("price")]
    by_source = {}
    for l in listings_raw:
        by_source[l["source"]] = by_source.get(l["source"], 0) + 1
    ts_rows.append(
        {
            "date": snap_date,
            "total": len(listings_raw),
            "with_price": len(prices),
            "median_price": float(np.median(prices)) if prices else None,
            "mean_price": float(np.mean(prices)) if prices else None,
            **{f"src_{k}": v for k, v in by_source.items()},
        }
    )

ts = pd.DataFrame(ts_rows)
ts["date"] = pd.to_datetime(ts["date"])
ts = ts.sort_values("date")
print(ts[["date", "total", "median_price"]].tail(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Total listings over time
axes[0].plot(ts["date"], ts["total"], marker="o", markersize=4, color="steelblue")
axes[0].fill_between(ts["date"], ts["total"], alpha=0.15, color="steelblue")
axes[0].set_ylabel("Total CPO listings")
axes[0].set_title("CPO Market Trends")
axes[0].grid(True, alpha=0.4)

# Median price over time
price_ts = ts[ts["median_price"].notna()]
axes[1].plot(price_ts["date"], price_ts["median_price"] / 1000, marker="o", markersize=4, color="darkorange")
axes[1].fill_between(price_ts["date"], price_ts["median_price"] / 1000, alpha=0.15, color="darkorange")
axes[1].set_ylabel("Median price (SGD '000)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Per-source listing counts over time
src_cols = [c for c in ts.columns if c.startswith("src_")]
fig, ax = plt.subplots(figsize=(13, 5))
for col in src_cols:
    src_name = col.replace("src_", "")
    label = SOURCE_LABELS.get(src_name, src_name)
    ax.plot(ts["date"], ts[col].fillna(0), marker="o", markersize=3, label=label)
ax.set_ylabel("Listings")
ax.set_xlabel("Date")
ax.set_title("CPO Listings per Dealer Over Time")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Brand Summary Table

In [ ]:
brand_stats = (
    clean.groupby("brand")
    .agg(
        count=("title", "count"),
        median_price=("price", "median"),
        median_age=("age_years", "median"),
        median_mileage=("mileage_km", "median"),
        median_annual_km=("annual_mileage_km", "median"),
        median_ppa=("price_per_age_year", "median"),
    )
    .reset_index()
)
brand_stats = brand_stats[brand_stats["count"] >= 3].sort_values("median_price")

bs = brand_stats.copy()
bs["median_price"] = bs["median_price"].apply(lambda x: f"${x:,.0f}" if pd.notna(x) else "—")
bs["median_age"] = bs["median_age"].apply(lambda x: f"{x:.1f} yr" if pd.notna(x) else "—")
bs["median_mileage"] = bs["median_mileage"].apply(lambda x: f"{x/1000:.0f}K km" if pd.notna(x) else "—")
bs["median_annual_km"] = bs["median_annual_km"].apply(lambda x: f"{x/1000:.0f}K km/yr" if pd.notna(x) else "—")
bs["median_ppa"] = bs["median_ppa"].apply(lambda x: f"${x/1000:.0f}K/yr" if pd.notna(x) else "—")
bs.columns = ["Brand", "Listings", "Median Price", "Median Age", "Median Mileage", "Annual km", "$/yr of age"]
print(bs.to_string(index=False))

## Summary & Caveats

### Data notes
- **Brand field is missing for most sources** — brand is extracted from the listing title. Occasionally the first word of the title is a year (e.g., "2019 Audi…"), which is handled automatically, but edge cases may mislabel.
- **Mileage coverage** is ~80%: Das WeltAuto and CarChoice often omit mileage.
- **Reg date coverage** is ~90%; a few listings only show a year.
- **No depreciation data** — unlike SGCarMart used-car listings, CPO sites don't publish official depreciation figures, so COE-adjusted metrics are not available.
- **Warranty** field is currently unpopulated across all scraped sources.

### Metrics defined
| Metric | Meaning |
|---|---|
| Price per year of age | Asking price ÷ car's age in years. Lower = more age absorbed per dollar. |
| Price per 10k km | Asking price ÷ (mileage ÷ 10,000). Lower = more km absorbed per dollar. |
| Annual mileage | Total mileage ÷ age. Proxy for usage intensity. |
| Composite score | Average of both normalised metrics. 0 = best, 1 = worst. |